# Part 6 — Cross-Species Workflows: Humanization

*Last updated:* 2026-11-12

This tutorial shows a practical **humanization** workflow: mapping mouse/pig genes into a human gene space so you can run
human-centric downstream analyses (pathways, marker lists, integration, annotation).

**Learning objectives**
- Understand what humanization is (and when it is appropriate).
- Run a step-by-step mouse → human and pig → human mapping.
- Validate results and handle 1→n orthology ambiguity explicitly.
- Prepare outputs in a tidy, analysis-friendly format for comparative workflows.

> **Warning:** Orthology is not always one-to-one. This notebook focuses on making ambiguity visible and manageable.


## 6.1 — Humanization workflow (what it is, and when to use it)

This notebook shows a **reproducible, step-by-step pipeline**:

1. **Within-species cleanup (IDTrack)**
   - Map mouse/pig identifiers to consistent Ensembl gene IDs within that species.

2. **Cross-species mapping (orthologs)**
   - Map mouse/pig Ensembl gene IDs to human ortholog Ensembl gene IDs.

3. **Human naming (optional, IDTrack)**
   - Convert human Ensembl gene IDs into HGNC symbols (or other human namespaces).

This notebook does **not** decide the ‘correct’ ortholog for you in complex families — it shows how to
surface the candidates and (optionally) score them with sequence-based heuristics.


### 6.1.1 Pre-requisites

### 6.1.1.1 IDTrack graphs
You should have built graphs for the organisms you use:
- `mus_musculus` and/or `sus_scrofa`
- `homo_sapiens`

### 6.1.1.2 Optional dependencies for ortholog utilities
Ortholog utilities require optional packages. Install one of:
- `pip install idtrack[ortholog]`
- or `pip install idtrack[all-external]`

If you don’t install these extras, the ortholog steps below are skipped with a message. The embedding features of section 6.1.4 also
need `torch` and `transformers` (`pip install idtrack[embeddings]`); with them installed, a protein language model is downloaded from
Hugging Face on first use, and without them those features are left empty.


In [1]:
# Check optional dependency status (ortholog utilities require gget + biopython)
from __future__ import annotations

from idtrack import _external_mappers

dep_status = _external_mappers.check_optional_dependencies(warn=True)
ORTHOLOG_OK = dep_status.get('gget', False) and dep_status.get('biopython', False)

print('Optional dependency status:', dep_status)
print('Ortholog utilities available:', ORTHOLOG_OK)


Optional dependency status: {'gget': True, 'mygene': True, 'pybiomart': True, 'gprofiler-official': True, 'biopython': True}
Ortholog utilities available: True


### 6.1.2 Step-by-step: mouse → human

### 6.1.2.1 Convert a mouse identifier to a mouse Ensembl gene ID (IDTrack)

Start with whatever you have (often an MGI symbol). Convert to a **base Ensembl gene ID** in mouse.


In [2]:
import os
from pathlib import Path
import idtrack

LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

api_mouse = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api_mouse.configure_logger()

mouse_name, mouse_latest = api_mouse.resolve_organism('mouse')
MOUSE_SNAPSHOT = 115  # pinned (mouse_latest is Ensembl's newest release)
api_mouse.build_graph(organism_name=mouse_name, snapshot_release=MOUSE_SNAPSHOT)

mouse_query = 'Trp53'  # example MGI symbol; replace with your gene
mouse_to_ensembl = api_mouse.convert_identifier(
    mouse_query,
    to_release=MOUSE_SNAPSHOT,
    final_database='base_ensembl_gene',
)
mouse_to_ensembl


2026-11-12 15:22:50 INFO:idtrack._verify_organism: Ensembl Rest API query to get the organism names and associated releases.


{'query_id': 'Trp53',
 'graph_id': 'Trp53',
 'match': 'exact',
 'known_versions': [],
 'from_release': 115,
 'from_release_inferred': True,
 'from_release_given': None,
 'from_release_fallback': False,
 'from_databases': ['MGI Symbol',
  'NCBI gene (formerly Entrezgene)',
  'UniProtKB Gene Name'],
 'to_release': 115,
 'strategy': 'best',
 'final_database': 'base_ensembl_gene',
 'target_id': ['ENSMUSG00000059552'],
 'last_node': [['ENSMUSG00000059552.14', 'ENSMUSG00000059552']],
 'ambiguous': False,
 'tie': False,
 'alternative_locus_genes': [],
 'no_corresponding': False,
 'no_conversion': False,
 'no_target': False,
 'no_target_genes': [],
 'error': None}

The `target_id` entry is your mouse Ensembl gene ID (without version, because of `final_database='base_ensembl_gene'`).
If you get several, you are in a 1→n case (`tie=True` when they are equal in all evidence) — you may need to decide how to handle it;
the first entry is the first of the best answer in a fixed order (by gene id), not a choice by evidence.


### 6.1.2.2 Find human ortholog(s) (ortholog utilities)

We use Bgee orthologs via `gget` (optional dependency).


In [3]:
human_ensembl_gene_ids = []
mouse_ensembl_gene_id = mouse_to_ensembl['target_id'][0] if mouse_to_ensembl.get('target_id') else None

if mouse_ensembl_gene_id is None:
    print('No mouse Ensembl gene ID found; check your input ID and mouse graph/YAML.')
elif not ORTHOLOG_OK:
    print('Ortholog utilities are not available (install extras: `pip install idtrack[ortholog]`).')
else:
    from idtrack._external_mappers import get_ortholog_table, get_ortholog_ids_for_species

    ortholog_df = get_ortholog_table(mouse_ensembl_gene_id, verbose=True)

    human_ensembl_gene_ids = sorted(get_ortholog_ids_for_species(ortholog_df, target_species='human'))

human_ensembl_gene_ids


15:22:53 - INFO - Getting species ID for gene ENSMUSG00000059552 from Bgee


15:22:53 - INFO - Getting orthologs for gene ENSMUSG00000059552 from Bgee


['ENSG00000141510']

### 6.1.2.3 Convert human Ensembl IDs into HGNC symbols (IDTrack)

Now we switch to the human graph and convert the human Ensembl IDs into HGNC symbols (optional but common).


In [4]:
if not human_ensembl_gene_ids:
    print('No human ortholog IDs available; skipping human conversion step.')
else:
    api_human = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
    api_human.configure_logger()

    human_name, human_latest = api_human.resolve_organism('human')
    HUMAN_SNAPSHOT = 115  # pinned
    api_human.build_graph(organism_name=human_name, snapshot_release=HUMAN_SNAPSHOT)

    # Convert all candidate orthologs (if many-to-many, you will see it here)
    human_results = api_human.convert_identifier_multiple(
        list(human_ensembl_gene_ids),
        to_release=HUMAN_SNAPSHOT,
        final_database='HGNC Symbol',
        verbose=False,
    )
    for r in human_results:
        print(r['query_id'], '->', r['target_id'], '| tie:', r['tie'])


ENSG00000141510 -> ['TP53'] | tie: False


### 6.1.3 Step-by-step: pig → human (same pattern)

Replace `api_mouse` with a pig API instance and start from your pig identifiers.
Many pig pipelines start from Ensembl IDs or Entrez IDs; adjust `final_database` accordingly.


In [5]:
api_pig = idtrack.API(local_repository=str(LOCAL_REPOSITORY))
api_pig.configure_logger()

pig_name, pig_latest = api_pig.resolve_organism('pig')  # 'pig' is sus_scrofa
PIG_SNAPSHOT = 115  # pinned
api_pig.build_graph(organism_name=pig_name, snapshot_release=PIG_SNAPSHOT)

pig_query = 'TP53'  # example; replace with your pig gene symbol or ID
pig_to_ensembl = api_pig.convert_identifier(
    pig_query,
    to_release=PIG_SNAPSHOT,
    final_database='base_ensembl_gene',
)
pig_to_ensembl


{'query_id': 'TP53',
 'graph_id': 'TP53',
 'match': 'exact',
 'known_versions': [],
 'from_release': 115,
 'from_release_inferred': True,
 'from_release_given': None,
 'from_release_fallback': False,
 'from_databases': ['NCBI gene (formerly Entrezgene)',
  'UniProtKB Gene Name',
  'VGNC Symbol'],
 'to_release': 115,
 'strategy': 'best',
 'final_database': 'base_ensembl_gene',
 'target_id': ['ENSSSCG00000017950'],
 'last_node': [['ENSSSCG00000017950.5', 'ENSSSCG00000017950']],
 'ambiguous': False,
 'tie': False,
 'alternative_locus_genes': [],
 'no_corresponding': False,
 'no_conversion': False,
 'no_target': False,
 'no_target_genes': [],
 'error': None}

From here, reuse the same ortholog + human conversion steps as in the mouse section.


### 6.1.4 Advanced: choose among multiple ortholog candidates

When you have multiple orthologs, IDTrack can optionally compute additional features for ranking.
This is for advanced use and requires extra dependencies.


In [6]:
# Example (advanced): compute sequence-based alignment features for each ortholog candidate
if mouse_ensembl_gene_id is None or not ORTHOLOG_OK:
    print('Skipped: no mouse Ensembl gene ID, or the ortholog utilities are not installed.')
    features = None
else:
    from idtrack._external_mappers import align_ortholog_pair_with_features

    features = align_ortholog_pair_with_features(mouse_ensembl_gene_id, target_species='human', verbose=True)
features


15:22:59 - INFO - Getting species ID for gene ENSMUSG00000059552 from Bgee


15:22:59 - INFO - Getting orthologs for gene ENSMUSG00000059552 from Bgee


[INFO] Found 1 ortholog(s) for 'ENSMUSG00000059552' in target species 'human' (canonical='hsapiens', genus='Homo', species='sapiens').


15:23:26 - INFO - Requesting amino acid sequence of the canonical transcript ENSMUST00000108658 of gene ENSMUSG00000059552 from UniProt.


[INFO] Aligning 'ENSMUSG00000059552' -> 'ENSG00000141510' ('human')


15:23:56 - INFO - Requesting amino acid sequence of the canonical transcript ENST00000269305 of gene ENSG00000141510 from UniProt.


15:23:56 - INFO - MUSCLE compiled. 


15:23:56 - INFO - MUSCLE aligning... 



muscle 5.2.linux64 [00617b]  791Gb RAM, 32 cores
Built Apr 13 2022 00:43:46
(C) Copyright 2004-2021 Robert C. Edgar.
https://drive5.com

Input: 2 seqs, avg length 392, max 393

00:00 7.9Mb  CPU has 32 cores, defaulting to 20 threads


00:00 134Mb   100.0% Calc posteriors
00:00 134Mb   100.0% UPGMA5         
15:23:56 - INFO - MUSCLE alignment complete. Alignment time: 0.05 seconds


[WARN] Failed to compute embeddings for ENSG00000141510: transformers / torch not installed. Install them or call align_ortholog_pair_with_features(..., embedding_model_name=None).


{'ENSG00000141510': {'query_id': 'ENSMUSG00000059552',
  'target_id': 'ENSG00000141510',
  'target_species': 'human',
  'target_species_canonical': 'hsapiens',
  'target_species_genus': 'Homo',
  'target_species_species': 'sapiens',
  'alignment_aligned_query': 'MTAMEESQSDISLELPLSQETFSGLWKLLPPEDIL-PSP-HCMDDLLL-PQDVEEFFE---GPSEALRVSGAPAAQDPVTETPGPVAPAPATPWPLSSFVPSQKTYQGNYGFHLGFLQSGTAKSVMCTYSPPLNKLFCQLAKTCPVQLWVSATPPAGSRVRAMAIYKKSQHMTEVVRRCPHHERCSDGDGLAPPQHLIRVEGNLYPEYLEDRQTFRHSVVVPYEPPEAGSEYTTIHYKYMCNSSCMGGMNRRPILTIITLEDSSGNLLGRDSFEVRVCACPGRDRRTEEENFRKKEVLCPELPPGSAKRALPTCTSASPPQKKKPLDGEYFTLKIRGRKRFEMFRELNEALELKDAHATEESGDSRAHSSYLKTKKGQSTSRHKKTMVKKVGPDSD',
  'alignment_aligned_target': '---MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGPDEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAKSVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHERCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNSSCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELPPGST

### 6.1.5 Practical cautions (please read)

1. **Orthology is context-dependent**: paralogs, gene family expansions, and annotation differences matter.
2. **Do not silently pick one** in many-to-many cases without recording the rule you used.
3. **Record provenance**: snapshot releases + YAML configs + ortholog source/version.


## 6.2 — Comparative Analysis Preparation

Once you have humanized identifiers, you can prepare a comparative workflow that is **reproducible** and **auditable**.

Recommended preparation steps:

1. **Within each species:** harmonize identifiers into stable Ensembl gene IDs at a fixed snapshot boundary.
2. **Across species:** map to human orthologs, but **keep ambiguity visible** (store 1→n mappings as lists).
3. **Record provenance:** snapshot boundaries, assemblies, and the orthology source/method.
4. **Define a policy for ambiguous cases:** drop / keep-all / choose-best (and justify it).

A practical output format is a tidy table with one row per input gene and explicit columns for provenance.
The next cell shows a minimal schema you can reuse.


In [7]:
# Example: a tidy, audit-friendly mapping table schema

import pandas as pd

mapping_table = pd.DataFrame(
    [
        {
            'source_species': 'mus_musculus',
            'source_namespace': 'MGI Symbol',
            'source_id': 'Trp53',
            'human_ensembl_gene_id': 'ENSG00000141510',
            'human_hgnc_symbol': 'TP53',
            'orthology_candidates': ['ENSG00000141510'],
            'snapshot_release_human': None,
            'snapshot_release_source': None,
            'notes': 'Example row; fill with your real results.'
        },
        {
            'source_species': 'sus_scrofa',
            'source_namespace': 'Ensembl Gene ID',
            'source_id': 'ENSSSCG00000000001',
            'human_ensembl_gene_id': None,
            'human_hgnc_symbol': None,
            'orthology_candidates': [],
            'snapshot_release_human': None,
            'snapshot_release_source': None,
            'notes': 'Example 1→0 / not found; keep these rows for reporting.'
        },
    ]
)

mapping_table


,source_species,source_namespace,source_id,human_ensembl_gene_id,human_hgnc_symbol,orthology_candidates,snapshot_release_human,snapshot_release_source,notes
0,mus_musculus,MGI Symbol,Trp53,ENSG00000141510,TP53,[ENSG00000141510],None,None,Example row; fill with your real results.
1,sus_scrofa,Ensembl Gene ID,ENSSSCG00000000001,NaN,NaN,[],None,None,Example 1→0 / not found; keep these rows for r...
